# Checkpoint inference and evaluation

Evaluate a stopped or completed run without training. Load validation-selected fitting weights, or an exported probability model. The saved configuration supplies the original architecture and source-label semantics. Full-volume metrics preserve physical spacing and missing-annotation statuses. Use the **rnet** kernel.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Run folder or explicit checkpoint and cohort

For a stopped run, set `RUN_DIR`, `FOLD` and `EVALUATION_ROLE`. Choose `test` for fixed splits, `holdout` for outer CV, or `validation` for historical rat CV. The saved cohort and committed best weights are selected automatically. `CHECKPOINT_KIND="weights"` is the default; `"model"` loads the final `model.keras` export.

Alternatively set `EXPERIMENT_FILE`, `MODEL_FILE` and a split/explicit animal cohort directly. A `.weights.h5` file requires `CHECKPOINT_KIND="weights"`. Optional data/reference overrides relocate inputs without changing output label semantics. Every selected animal includes all timepoints. Evaluation writes to a fresh folder outside the training run.

`REFERENCE_OVERRIDE` defaults to the shared RARE reference (R, I, A), including evaluation of older checkpoints. The reference changes orientation without resizing to the reference image's shape. Older weights trained in a different orientation now receive a changed input convention; both the saved training reference and the evaluation override are retained in provenance.

Full-volume settings explicitly clean the brain mask and restore original annotation IDs. `VENTRICLE_MODE="auto"` separates the merged D1/rat ventricles and preserves MouseFull sides. Morphology thresholds are voxel counts. The historical fallback hemisphere refers to the side of array origin; missing predicted third-ventricle voxels raise an error when splitting is needed.


In [ ]:
from typing import Literal
from uuid import uuid4
from src.experiments.config import DataConfig, ExperimentConfig, PostprocessingConfig
from src.evaluation.inference import (
    SavedModelEvaluationConfig, best_checkpoint_settings, evaluate_saved_model,
    load_saved_evaluation, saved_prediction_manifest, prediction_comparison_figure,
)

RUN_DIR: Path | None = None  # results/<model>/<run>, including interrupted runs.
CHECKPOINT_KIND: Literal["model", "weights"] = "weights"
EXPERIMENT_FILE: Path | None = None
MODEL_FILE: Path | None = None  # .weights.h5 or exported model.keras
SPLIT_FILE: Path | None = None
SPLIT_ROLE: str = "test"
SUBJECT_IDS: tuple[str, ...] = ()
EVALUATION_ROLE: str = "test"  # CV: holdout; historical rat CV: validation.
VARIANT: str = "3drnet"
TRAINING_SEED: int = 42
FOLD: int = 0
DATA_OVERRIDE: DataConfig | None = None
REFERENCE_OVERRIDE: Path | None = REFERENCE
STRIDE_OVERRIDE: int | None = None
EVALUATE_PATCHES: bool = True  # Native saved objectives/metrics; no fitting.
SAVE_PREDICTIONS: bool = True
ORIGINAL_LABEL_IDS: bool = True
VENTRICLE_MODE: Literal["auto", "split", "keep"] = "auto"  # auto: split only the merged D1/rat vocabulary.
CLEAN_BRAIN_MASK: bool = True
USE_VENTRICLE_CENTROIDS: bool = True
DEFAULT_HEMISPHERE: Literal["right", "left"] = "right"
HOLES_MAX_AREA: int = 20000
OBJECT_MIN_AREA: int = 30000
LESION_VOXELS_THRESHOLD: int = 10
RUN_EVALUATION: bool = False
evaluation_id = uuid4().hex[:12]
EVALUATION_DIR: Path = REPO_ROOT / "outputs" / "inference" / evaluation_id

if RUN_DIR is not None:
    run_selection = best_checkpoint_settings(
        run_dir=RUN_DIR, fold=FOLD, role=EVALUATION_ROLE, output_dir=EVALUATION_DIR,
    )
    if EXPERIMENT_FILE is None:
        EXPERIMENT_FILE = run_selection.experiment_file
    if MODEL_FILE is None:
        MODEL_FILE = run_selection.model_file
        if CHECKPOINT_KIND == "model":
            MODEL_FILE = RUN_DIR / f"fold_{FOLD}" / "model.keras"
    if SPLIT_FILE is None and not SUBJECT_IDS:
        SUBJECT_IDS = run_selection.subject_ids
    VARIANT = run_selection.variant
    TRAINING_SEED = run_selection.seed

saved_config = None
if EXPERIMENT_FILE is not None:
    saved_config = ExperimentConfig.load(EXPERIMENT_FILE)
    print("Saved architecture:", saved_config.model.architecture)
    print("Output classes:", saved_config.labels.names)
    print("Patch / spacing:", saved_config.preprocessing.patch_size, saved_config.preprocessing.spacing_mm)
    print("Checkpoint:", MODEL_FILE, "kind:", CHECKPOINT_KIND)
else:
    print("Set RUN_DIR, or EXPERIMENT_FILE and MODEL_FILE, to inspect a checkpoint.")

postprocessing_settings = None
if saved_config is not None:
    if VENTRICLE_MODE not in ("auto", "split", "keep"):
        raise ValueError("VENTRICLE_MODE must be auto, split or keep")
    split_ventricles = VENTRICLE_MODE == "split"
    if VENTRICLE_MODE == "auto":
        merged_mapping = {0: 0, 1: 1, 3: 2, 13: 2, 21: 3}
        split_ventricles = saved_config.labels.mapping == merged_mapping

    postprocessing_settings = PostprocessingConfig(
        clean_brain_mask=CLEAN_BRAIN_MASK,
        fill_small_holes=True, holes_max_area=HOLES_MAX_AREA,
        remove_small_objects=True, object_min_area=OBJECT_MIN_AREA,
        split_ventricles=split_ventricles, use_centroids=USE_VENTRICLE_CENTROIDS,
        default_hemisphere=DEFAULT_HEMISPHERE,
        lesion_voxels_threshold=LESION_VOXELS_THRESHOLD,
    )
    print("Full-volume postprocessing:", postprocessing_settings)
    print("Restore original annotation IDs:", ORIGINAL_LABEL_IDS)


## Review the explicit selection

No default selects all registered scans. Prefer untouched fixed-split test or outer holdout animals. An external role requires a verified cohort that was not used for fitting or checkpoint selection. This notebook records the chosen role; it cannot infer the checkpoint's training history from animal names.

In [ ]:
import json

selected_ids: tuple[str, ...] = SUBJECT_IDS
if SPLIT_FILE is not None:
    if SUBJECT_IDS:
        raise ValueError("Choose a split file or explicit SUBJECT_IDS, not both")
    split_records = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
    selected_ids = tuple(split_records[SPLIT_ROLE])

print("Selected animals:", len(selected_ids))
print("Evaluation role:", EVALUATION_ROLE)
display(pd.DataFrame({"Subject": selected_ids}))

## Full-volume computation

Weights-only loading rebuilds the exact fitting graph from the saved configuration, loads all checkpoint weights, and exposes full softmax probabilities for inference. It performs no compilation, optimizer restoration or fit. Exported models still load with `compile=False`.

Execution checks input/head shapes and writes a fresh output folder with checkpoint kind/hash, source/evaluation configurations, selected animals, data contract, metrics, timings and optional original-grid ROI/brain masks. Training recovery files and completion flags are unchanged. The settings above apply historical brain cleanup and optional merged-ventricle separation before restoring original label IDs.


In [ ]:
evaluation = None
if RUN_EVALUATION:
    if EXPERIMENT_FILE is None or MODEL_FILE is None:
        raise ValueError("Set RUN_DIR or the experiment/checkpoint file paths before running")
    settings = SavedModelEvaluationConfig(
        experiment_file=EXPERIMENT_FILE, model_file=MODEL_FILE, output_dir=EVALUATION_DIR,
        subject_ids=selected_ids, role=EVALUATION_ROLE, variant=VARIANT,
        seed=TRAINING_SEED, fold=FOLD, save_predictions=SAVE_PREDICTIONS,
        data=DATA_OVERRIDE, reference=REFERENCE_OVERRIDE, stride=STRIDE_OVERRIDE,
        checkpoint_kind=CHECKPOINT_KIND, evaluate_patches=EVALUATE_PATCHES,
        original_label_ids=ORIGINAL_LABEL_IDS, postprocessing=postprocessing_settings,
    )
    evaluation = evaluate_saved_model(settings)
    print("Saved to:", evaluation.output_dir)
else:
    print("Set RUN_EVALUATION=True after selecting the checkpoint and cohort.")


## Measurements, statuses and failure cases

Review lesion failures alongside missing annotations and genuinely absent structures. Undefined values remain missing. These are raw scan-level rows; notebook 10 aggregates biological animals for uncertainty. Inspect status counts before sorting scores.

In [ ]:
if evaluation is not None:
    from src.evaluation.classification import classification_report, healthy_lesion_report
    classifier_results = classification_report(evaluation.classifier_predictions)
    if not classifier_results.summary.empty:
        display(classifier_results.summary)
        display(classifier_results.confusion)
    sham_animals, sham_summary = healthy_lesion_report(evaluation.metrics)
    if not sham_summary.empty:
        display(sham_summary)
    display(evaluation.patch_metrics)
    print("Evaluation workbook:", evaluation.output_dir / "evaluation_metrics.xlsx")
    display(evaluation.manifest)
    display(evaluation.timings)
    display(evaluation.metrics.groupby(["Region", "Metric", "Status"]).size().rename("Rows").reset_index())
    dice = evaluation.metrics.loc[evaluation.metrics["Metric"] == "Dice"]
    display(dice.sort_values("Value", na_position="last"))
    figure = px.scatter(
        dice, x="Scan", y="Value", color="Region", symbol="Status",
        hover_data=["Subject", "Dataset"], title="Full-volume Dice by scan",
    )
    figure.show()

## Explore MRI, ground truth and prediction

Use the new evaluation above, or set `VIEW_EVALUATION_DIR` to a previously saved evaluation folder. Reopening results does not recompute predictions. The inventory includes animals, timepoints and folds. Native Plotly menus select loaded scans/planes; a slider selects slices. Both overlays share categorical colors and follow the saved reference orientation. Missing GT annotations are identified in the title.

`VIEW_SCAN_INDICES=None` loads up to three rows; choose explicit indices for other scans. `VIEW_REGION` selects one foreground original annotation ID. Previews sample up to 64 slices and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice or increase `VIEW_MAX_PIXELS`.


In [ ]:
VIEW_EVALUATION_DIR: Path | None = None
VIEW_SCAN_INDICES: tuple[int, ...] | None = None
VIEW_REGION: int | None = None
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192

viewer_evaluation = globals().get("evaluation")
if VIEW_EVALUATION_DIR is not None:
    viewer_evaluation = load_saved_evaluation(VIEW_EVALUATION_DIR, reference=REFERENCE)

if viewer_evaluation is None:
    print("Run evaluation, or set VIEW_EVALUATION_DIR to existing saved results.")
else:
    viewer_role = str(viewer_evaluation.metrics["DatasetRole"].iloc[0])
    prediction_views = saved_prediction_manifest(
        data=viewer_evaluation.data, run_dir=viewer_evaluation.output_dir, role=viewer_role,
    )
    display(prediction_views)
    if prediction_views.empty:
        print("No prediction files were saved. Enable SAVE_PREDICTIONS when evaluating.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=viewer_evaluation.data, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            max_slices=VIEW_MAX_SLICES, max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")
